# Uten IMP 全库关联关系审计

本 notebook 只读连接隔离数据库 uten_imp_relation_verify_20260801，复核 V182 后的关系元数据、迁移对账、问题分组及历史成本风险。

In [1]:
import json
import subprocess

CONTAINER = "uten-imp-postgres"
DATABASE = "uten_imp_relation_verify_20260801"

def query(sql):
    command = ["docker", "exec", CONTAINER, "psql", "-U", "uten", "-d", DATABASE,
               "-v", "ON_ERROR_STOP=1", "-P", "pager=off", "-F", "|", "-Atc", sql]
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return [line.split("|") for line in result.stdout.splitlines() if line]

print(f"database={DATABASE}")

database=uten_imp_relation_verify_20260801


## 1. 结构规模

FK 只能证明物理行存在，不能阻止软删除或区分历史快照。

In [2]:
metadata_sql = """
SELECT 'tables',count(*) FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace WHERE c.relkind='r' AND n.nspname='public';
SELECT 'fk_total',count(*) FROM pg_constraint c JOIN pg_namespace n ON n.oid=c.connamespace WHERE c.contype='f' AND n.nspname='public';
SELECT 'not_valid_total',count(*) FROM pg_constraint c JOIN pg_namespace n ON n.oid=c.connamespace WHERE n.nspname='public' AND NOT c.convalidated;
SELECT 'soft_delete_tables',count(*) FROM information_schema.columns WHERE table_schema='public' AND column_name='is_deleted';
SELECT 'dual_key_groups',count(*) FROM (
 SELECT table_name,regexp_replace(column_name,'(_legacy)?_id$','') stem FROM information_schema.columns
 WHERE table_schema='public' AND column_name LIKE '%_id'
 GROUP BY table_name,regexp_replace(column_name,'(_legacy)?_id$','')
 HAVING bool_or(data_type='uuid') AND bool_or(data_type<>'uuid')) s;
"""
for key, value in query(metadata_sql):
    print(f"{key:20s} {value}")

tables               189
fk_total             579
not_valid_total      19
soft_delete_tables   125
dual_key_groups      100


## 2. V182 reconciliation

status=SUCCESS 只表示 SQL 已提交；生产验收取决于 reconciliation_status 和问题计数。

In [3]:
run_json = query("SELECT row_to_json(r) FROM (SELECT * FROM legacy_migration_runs ORDER BY started_at DESC LIMIT 1) r")[0][0]
print(json.dumps(json.loads(run_json), ensure_ascii=False, indent=2))

{
  "run_id": "e957286e-e543-4dbf-8736-0d24fc57d081",
  "target": "schema:V182-live-master-relationships",
  "status": "SUCCESS",
  "started_at": "2026-08-01T14:46:29.855592+08:00",
  "finished_at": "2026-08-01T14:46:29.855592+08:00",
  "exit_code": 0,
  "database_user": "uten",
  "migration_mode": "INCREMENTAL",
  "export_manifest_sha256": null,
  "checksum_manifest_sha256": null,
  "migration_repository_commit": null,
  "migration_script_sha256": null,
  "mapping_version": "v182-live-master-uuid-v1",
  "reconciliation_status": "FAILED",
  "reconciliation_summary": {
    "scope": "current database relationship rows at Flyway execution time",
    "issueCount": 320,
    "failedMetricCount": 3,
    "productionAcceptance": false,
    "sourceReferenceCount": 347012,
    "postImportReconciliationRequired": true
  },
  "rejected_count": 320
}


## 3. Reject 与覆盖率

每个问题必须保留在 reject 表，不允许用猜测或 waiver 把迁移状态改成通过。

In [4]:
reject_sql = """
SELECT reason_code,COUNT(*) FROM legacy_migration_rejects
WHERE run_id=(SELECT run_id FROM legacy_migration_runs ORDER BY started_at DESC LIMIT 1)
GROUP BY reason_code ORDER BY reason_code
"""
for reason, count in query(reject_sql):
    print(f"{reason:42s} {count}")

coverage_sql = """
SELECT metric,expected_value,actual_value,passed,detail
FROM legacy_migration_reconciliation_items
WHERE run_id=(SELECT run_id FROM legacy_migration_runs ORDER BY started_at DESC LIMIT 1)
ORDER BY metric,source_entity
"""
print("\ncoverage")
for metric, expected, actual, passed, detail in query(coverage_sql):
    print(f"{metric:42s} expected={expected} actual={actual} passed={passed} | {detail}")

GOODS_BOM_COLOR_LEGACY_UNMAPPED            134
GOODS_BOM_NON_POSITIVE_QTY                 112
GOODS_BOM_SUPPLIER_LEGACY_UNMAPPED         23
GOODS_COLOR_LEGACY_UNMAPPED                3
GOODS_DEFAULT_SUPPLIER_LEGACY_UNMAPPED     12
GOODS_MOULD_LEGACY_UNMAPPED                24
STOCK_MAKER_UUID_ORPHAN                    1
STOCK_WORKER_UUID_ORPHAN                   11

coverage
deterministic_b_worker_reference_coverage  expected=42 actual=42 passed=t | worker_legacy_id is B_Worker.ID; maker/approver legacy IDs are Sys_Operator snapshots and are excluded
deterministic_nonzero_reference_coverage   expected=78875 actual=78718 passed=f | Includes active and V181-isolated historical BOM rows; no row is reactivated
deterministic_nonzero_reference_coverage   expected=68413 actual=68374 passed=f | UUID must resolve to a master row whose legacy_id equals the retained source value
recorded_relationship_issue_count          expected=0 actual=320 passed=f | Every issue remains reviewable in legacy_mi

## 4. 历史成本漂移风险

总账与成本报表仍以历史数量乘当前 goods.c_total。现有 cost_amount 多数为零，且新单可由客户端提交，因此还不是权威 COGS 快照。

In [5]:
cost_sql = """
SELECT COUNT(*),
 COUNT(*) FILTER (WHERE i.cost_amount IS NULL),
 COUNT(*) FILTER (WHERE i.cost_amount=0),
 COUNT(*) FILTER (WHERE i.cost_amount IS NOT NULL AND i.cost_amount<>0),
 COUNT(*) FILTER (WHERE i.cost_amount IS NOT NULL AND i.qty<>0
   AND abs(i.cost_amount-(i.qty*COALESCE(g.c_total,0)))>0.01)
FROM sales_shipment_items i JOIN sales_shipments d ON d.id=i.shipment_id
LEFT JOIN goods g ON g.id=i.goods_id
WHERE d.status=1 AND d.is_deleted=false AND i.is_deleted=false
"""
labels = ["approved_items","null_cost","zero_cost","populated_cost","differs_from_current_cost"]
for label, value in zip(labels, query(cost_sql)[0]):
    print(f"{label:30s} {value}")

approved_items                 90881
null_cost                      0
zero_cost                      76571
populated_cost                 14310
differs_from_current_cost      88413


## 5. 结论

- 当前 BOM、订单等实时关系应通过 UUID JOIN 显示主档新名称。
- 审核单重打印、价格、汇率、税额、成本、审核人与来源单号必须使用不可变快照。
- V182 可执行，但 reconciliation 仍为 FAILED，不能进入生产验收。
- 全量 CSV 重导缺同次导出的 manifest/checksum，不得绕过该门禁。